In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score

In [3]:
# read the CSV file into a DataFrame
df = pd.read_csv("/home/sk/Projects/Astrophysics/data/ALTAS.csv")
print("Original dataset shape:", df.shape)

Original dataset shape: (1311, 122)


In [4]:
# extratcting the features and target variable
features = [
    "Sp2",
    "flux_ap2_36",
    "flux_ap2_45",
    "flux_ap2_58",
    "flux_ap2_80",
    "MAG_APER_4_G",
    "MAG_APER_4_R",
    "MAG_APER_4_I",
    "MAG_APER_4_Z"
]
target = "z"

# converting the features and target variable to numpy arrays
X = df[features].to_numpy(dtype=np.float32)
y = df[target].to_numpy(dtype=np.float32)

# splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("DATA PARTITION")
print("Total sources   :", len(X))
print("Training sources:", len(X_train))
print("Testing sources :", len(X_test))


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.

    A source is an outlier when:
    |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    error = np.abs(y_pred - y_true)

    outliers = (
        error >
        0.15 * (1 + y_true)
    )

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.

# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(metric="euclidean"))
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}


# defining the K-Fold cross-validation strategy
cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=10
)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)


# show results for all k values
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results["eta_0.15"] = (-cv_results["mean_test_score"])
cv_results = cv_results[
    [
        "param_knn__n_neighbors",
        "eta_0.15"
    ]
]
print("CROSS-VALIDATION RESULTS")
print(cv_results.to_string(index=False))

DATA PARTITION
Total sources   : 1311
Training sources: 917
Testing sources : 394
Best k: 6
Best CV eta_0.15: 0.08718346870520784
CROSS-VALIDATION RESULTS
 param_knn__n_neighbors  eta_0.15
                      2  0.100299
                      3  0.092630
                      4  0.099164
                      5  0.092642
                      6  0.087183
                      7  0.092642
                      8  0.090456
                      9  0.094828
                     10  0.099212
                     11  0.100287
                     12  0.101386
                     13  0.104646
                     14  0.107943
                     15  0.103595
                     16  0.104682
                     17  0.104694
                     18  0.109054
                     19  0.113426


In [5]:
# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)


print(f"Training sources: {len(y_train)}")
print(f"Testing sources: {len(y_test)}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Best k: {best_k}")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Training sources: 917
Testing sources: 394
Number of features: 9
Best k: 6
eta_0.15: 7.36%
eta_2sigma: 5.84%
sigma: 0.1024
sigma_NMAD: 0.0555
R²: 0.6404
MSE: 0.0613
RMSE: 0.2475
MAE: 0.1078


According to the paper an object is an outlier when $|z_{pred} - z_{spec}| > 0.15 * (1 + z_{spec})$ .We return the FRACTIO{N of outliers.Lower is better. Normalized residuals used in paper is $delta_z = (z_{spec} - z_{photo}) / 1 + z_{spec} $.

sigma_NMAD calculation :
 MAD = median($|\delta_z - median(\delta z)|$)
 sigma_NMAD = 1.4826 × MAD


# REu2

In [6]:
from knn_eu import knn_eu

df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")  # training dataset
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")  # testing dataset

REu2 = knn_eu(df_elais, df_cdfs)

DATA PARTITION
Training sources: 495
Testing sources : 816
Number of features: 9

BEST MODEL
Best k: 5
Best CV eta_0.15: 0.10710204081632653

CROSS-VALIDATION RESULTS
 param_knn__n_neighbors  eta_0.15
                      2  0.109020
                      3  0.116980
                      4  0.123143
                      5  0.107102
                      6  0.107102
                      7  0.113224
                      8  0.117224
                      9  0.119306
                     10  0.117306
                     11  0.125388
                     12  0.127429
                     13  0.135510
                     14  0.141592
                     15  0.149673
                     16  0.149673
                     17  0.151673
                     18  0.143551
                     19  0.147592
                     20  0.147592
                     21  0.139388
                     22  0.137388
                     23  0.133347
                     24  0.133306
                 

# REu3

In [7]:
REu3 = knn_eu(df_cdfs, df_elais)

DATA PARTITION
Training sources: 816
Testing sources : 495
Number of features: 9

BEST MODEL
Best k: 11
Best CV eta_0.15: 0.08706714844926226

CROSS-VALIDATION RESULTS
 param_knn__n_neighbors  eta_0.15
                      2  0.095694
                      3  0.098148
                      4  0.092005
                      5  0.109214
                      6  0.103056
                      7  0.094460
                      8  0.094444
                      9  0.096868
                     10  0.089506
                     11  0.087067
                     12  0.095649
                     13  0.098103
                     14  0.096899
                     15  0.095679
                     16  0.093240
                     17  0.093225
                     18  0.091960
                     19  0.093210
                     20  0.094444
                     21  0.095664
                     22  0.094444
                     23  0.098148
                     24  0.098148
                